# 🛢️ Naija-Petro 32B — Fine-Tune, Evaluate & Deploy on Hugging Face

**Notebook 5 of 5 · Naija-Petro project** · Qwen3-32B fine-tuned on 20K petroleum-engineering instruction–response pairs.

**Why Qwen3-32B?** On an A100 (40/80 GB), QLoRA 4-bit uses only ~22 GB VRAM, Qwen3 leads current fine-tuning benchmarks, and it ships under Apache-2.0. This is the highest-quality variant; the lighter 8B (notebook 4) is the one deployed in the RAG app.

| Step | Tool |
|---|---|
| Fine-tuning | Unsloth QLoRA (2× faster) |
| Tracking | Weights & Biases |
| Hub | HF Hub (16-bit + GGUF) |
| Demo | HF Spaces (Gradio) |
| Eval | Custom 30-question petroleum eval + GainEnergy benchmark |

---
## 0. Install & GPU Check

In [ ]:
!pip install triton==3.1.0 --no-deps -q

# Force restart runtime to clear cached imports
import os
os.kill(os.getpid(), 9)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 209.6/209.6 MB 11.6 MB/s eta 0:00:00


In [ ]:
# INSTALL — pin triton to avoid version conflicts
!pip install --no-deps bitsandbytes accelerate xformers peft trl triton==3.1.0 cut_cross_entropy unsloth_zoo 2>/dev/null
!pip install sentencepiece protobuf "datasets>=3.4.1" huggingface_hub hf_transfer 2>/dev/null
!pip install --no-deps unsloth 2>/dev/null
!pip install wandb jsonlines rouge-score nltk gradio 2>/dev/null

import torch
gpu = torch.cuda.get_device_name(0)
vram = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"GPU: {gpu} | VRAM: {vram:.1f} GB | CUDA: {torch.version.cuda}")

if vram >= 35:
    RECOMMENDED_MODEL = "unsloth/Qwen3-32B"
    print(f"A100 detected -> Qwen3-32B (~22GB QLoRA)")
elif vram >= 20:
    RECOMMENDED_MODEL = "unsloth/Qwen3-14B"
    print(f"L4/A10 detected -> Qwen3-14B (~12GB QLoRA)")
else:
    RECOMMENDED_MODEL = "unsloth/Qwen3-8B"
    print(f"T4 detected -> Qwen3-8B (~8GB QLoRA)")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 42.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 91.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 528.8/528.8 kB 51.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 401.6/401.6 kB 44.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 97.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.7/53.7 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.7/62.7 MB 38.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for rouge-score: filename=rouge_score-0.1.2-py3-none-any.whl size=24934 sha256=a8d046a40cd2438dd9880820040ae767782f9e77ce07da65b5de74ffe90057ac
  Stored in directory: /root/.cache/pip/wheels/85/9d/af/01feefbe7d55ef5468796f0c68225b6788e85d9d0a281e7a70
Successfully built rouge-score
GPU: NVIDIA A100-SXM4-80GB | VRAM: 85.1 GB | CUDA: 12.8
A100 detected -> Qwen3-32B (~22GB QLoRA)


---
## 1. Configuration

In [ ]:
from pathlib import Path
from google.colab import userdata

# --- EDIT THESE ---
HF_USERNAME       = userdata.get('HF_USERNAME')
HF_MODEL_NAME     = "naija-petro"
HF_REPO_ID        = f"{HF_USERNAME}/{HF_MODEL_NAME}"
HF_GGUF_REPO      = f"{HF_REPO_ID}-GGUF"
HF_SPACE_REPO      = f"{HF_USERNAME}/{HF_MODEL_NAME}-chat"

# Base Model (auto-selected or override)
BASE_MODEL         = RECOMMENDED_MODEL
MAX_SEQ_LENGTH     = 2048
LOAD_IN_4BIT       = True

# LoRA
LORA_R, LORA_ALPHA, LORA_DROPOUT = 64, 128, 0.0

# ══════════════════════════════════════════════════════════════
# Training — OPTIMISED FOR A100 80GB
# ══════════════════════════════════════════════════════════════
# Old (40GB): BATCH_SIZE=8,  GRAD_ACCUM=2,  effective=16, ~880 steps
# New (80GB): BATCH_SIZE=32, GRAD_ACCUM=2,  effective=64, ~220 steps → 4x fewer steps
#
# If you get OOM, step down:  BATCH_SIZE=24 (~292 steps, 3x faster)
# If still OOM:               BATCH_SIZE=16 (~440 steps, 2x faster)
NUM_EPOCHS         = 2
BATCH_SIZE         = 8          # 8  → 32 (fills 80GB GPU)
GRAD_ACCUM_STEPS   = 8           # keep 2 for effective batch=64 (good for convergence)
LEARNING_RATE      = 2e-4        # 1e-4 → 2e-4 (scale up with 4x larger effective batch)
WARMUP_RATIO       = 0.05
LR_SCHEDULER       = "cosine"
WEIGHT_DECAY       = 0.01
LOGGING_STEPS      = 10          # 25 → 10 (fewer steps total, log more often)
SAVE_STEPS         = 50          # 500 → 50 (save to Drive every ~50 steps)
EVAL_STEPS         = 100         # 500 → 100 (eval more often with fewer total steps)
SEED               = 42

# Paths — checkpoints on DRIVE (survive crashes)
DRIVE_BASE         = Path("/content/drive/MyDrive/petroleum_corpus")
DATASET_PATH       = DRIVE_BASE / "processed" / "final_alpaca_format.jsonl"
OUTPUT_DIR         = str(DRIVE_BASE / "checkpoints" / HF_MODEL_NAME)
GGUF_QUANTS        = ["q4_k_m", "q5_k_m", "q8_0"]

# W&B
WANDB_PROJECT, USE_WANDB = "naija-petro", True

# System prompt
SYSTEM_PROMPT = ("You are Naija-Petro, an expert petroleum engineering AI assistant. "
    "You provide precise, technically accurate answers covering drilling, "
    "reservoir engineering, production, completions, EOR, well testing, "
    "and petroleum geoscience. Include equations, units, and practical considerations.")

print(f"Config: {BASE_MODEL} | LoRA r={LORA_R} | {HF_REPO_ID}")
print(f"Batch: {BATCH_SIZE} x {GRAD_ACCUM_STEPS} = {BATCH_SIZE * GRAD_ACCUM_STEPS} effective")
print(f"Checkpoints: {OUTPUT_DIR}")
print(f"Estimated VRAM: ~{21.5 + BATCH_SIZE * 1.0:.0f} GB / 80 GB")

Config: unsloth/Qwen3-32B | LoRA r=64 | Shinzmann/naija-petro
Batch: 8 x 8 = 64 effective
Checkpoints: /content/drive/MyDrive/petroleum_corpus/checkpoints/naija-petro
Estimated VRAM: ~30 GB / 80 GB


---
## 2. Authenticate

In [ ]:
from google.colab import drive, userdata
drive.mount('/content/drive')
assert DATASET_PATH.exists(), f"Dataset not found: {DATASET_PATH}"

from huggingface_hub import login
try:
    login(token=userdata.get('HF_TOKEN'))
    print("HF auth via secret")
except: login()

if USE_WANDB:
    import wandb
    try: wandb.login(key=userdata.get('WANDB_API_KEY'))
    except: wandb.login()
else:
    import os; os.environ["WANDB_DISABLED"] = "true"
print("All authenticated")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
HF auth via secret


/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: chidi-ashinze (naija-petro) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


All authenticated


In [ ]:
!pip install triton==3.1.0 --no-deps -q

---
## 3. Load Model + LoRA

In [ ]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL, max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=LOAD_IN_4BIT, dtype=None,
)
print(f"Loaded {BASE_MODEL}: {model.num_parameters():,} params, {torch.cuda.memory_allocated()/1e9:.1f}GB VRAM")

model = FastLanguageModel.get_peft_model(model,
    r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=LORA_DROPOUT,
    target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
    bias="none", use_gradient_checkpointing="unsloth", random_state=SEED, max_seq_length=MAX_SEQ_LENGTH,
)
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"LoRA: {trainable:,} trainable ({100*trainable/total:.2f}%) | VRAM: {torch.cuda.memory_allocated()/1e9:.1f}GB")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.3.7: Fast Qwen3 patching. Transformers: 5.0.0.
   \\   /|    NVIDIA A100-SXM4-80GB. Num GPUs = 1. Max memory: 79.251 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.0. CUDA Toolkit: 12.8. Triton: 3.1.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/707 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/237 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

unsloth/qwen3-32b-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.
Loaded unsloth/Qwen3-32B: 32,762,123,264 params, 19.3GB VRAM


Unsloth 2026.3.7 patched 64 layers with 64 QKV layers, 64 O layers and 64 MLP layers.


LoRA: 536,870,912 trainable (3.03%) | VRAM: 21.5GB


---
## 4. Load & Format Dataset

In [ ]:
from datasets import load_dataset

ds = load_dataset("json", data_files=str(DATASET_PATH), split="train")
print(f"Loaded {len(ds):,} samples | Columns: {ds.column_names}")

def format_to_chat(example):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": example["instruction"]},
    ]
    if example.get("input") and str(example["input"]).strip():
        messages[-1]["content"] += "\nContext: " + example["input"]
    messages.append({"role": "assistant",
        "content": example.get("output", example.get("response", ""))})
    return {"text": tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=False, enable_thinking=False)}

ds_fmt = ds.map(format_to_chat, num_proc=4, remove_columns=ds.column_names)
split = ds_fmt.train_test_split(test_size=0.05, seed=SEED)
ds_train, ds_eval = split["train"], split["test"]
print(f"Train: {len(ds_train):,} | Eval: {len(ds_eval):,}")
print(f"\nSample:\n{ds_fmt[0]['text'][:400]}")

Generating train split: 0 examples [00:00, ? examples/s]

Loaded 33,859 samples | Columns: ['instruction', 'input', 'output']


Map (num_proc=4):   0%|          | 0/33859 [00:00<?, ? examples/s]

Train: 32,166 | Eval: 1,693

Sample:
<|im_start|>system
You are Naija-Petro, an expert petroleum engineering AI assistant. You provide precise, technically accurate answers covering drilling, reservoir engineering, production, completions, EOR, well testing, and petroleum geoscience. Include equations, units, and practical considerations.<|im_end|>
<|im_start|>user
During a subsea production run, the subsea pressure gauge on a 7‑inch


In [ ]:
# ══════════════════════════════════════════════════════════════
# AUTO-SCALE: Adjust batch to fit whatever GPU Colab gives you
# ══════════════════════════════════════════════════════════════
EFFECTIVE_BATCH = 64  # always keep this constant for consistent training

vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
gpu_name = torch.cuda.get_device_name(0)

if vram_gb >= 70:
    BATCH_SIZE     = 8
    GRAD_ACCUM_STEPS = EFFECTIVE_BATCH // BATCH_SIZE  # 8
elif vram_gb >= 35:
    BATCH_SIZE     = 4
    GRAD_ACCUM_STEPS = EFFECTIVE_BATCH // BATCH_SIZE  # 16
elif vram_gb >= 20:
    BATCH_SIZE     = 2
    GRAD_ACCUM_STEPS = EFFECTIVE_BATCH // BATCH_SIZE  # 32
else:
    BATCH_SIZE     = 1
    GRAD_ACCUM_STEPS = EFFECTIVE_BATCH // BATCH_SIZE  # 64

print(f"GPU: {gpu_name} | VRAM: {vram_gb:.0f} GB")
print(f"Auto-scaled: batch={BATCH_SIZE} x accum={GRAD_ACCUM_STEPS} = {BATCH_SIZE * GRAD_ACCUM_STEPS} effective")
print(f"Checkpoints are compatible across all GPU sizes — same effective batch")

---
## 5. Train

In [ ]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

from trl import SFTTrainer, SFTConfig
import time
import glob

Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

args = SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=2,                # eval uses less VRAM
    gradient_accumulation_steps=GRAD_ACCUM_STEPS,
    learning_rate=LEARNING_RATE,
    lr_scheduler_type=LR_SCHEDULER,
    warmup_ratio=WARMUP_RATIO,
    weight_decay=WEIGHT_DECAY,
    max_seq_length=MAX_SEQ_LENGTH,
    torch_compile=False,                         # MUST be False — compile pre-allocates 18GB for loss
    packing=True,
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    logging_steps=LOGGING_STEPS,
    save_steps=SAVE_STEPS,
    eval_strategy="steps",
    eval_steps=EVAL_STEPS,
    save_total_limit=3,
    seed=SEED,
    report_to="wandb" if USE_WANDB else "none",
    run_name=f"naija-petro-r{LORA_R}",
    optim="adamw_8bit",
    dataset_text_field="text",
)

trainer = SFTTrainer(
    model=model, tokenizer=tokenizer,
    train_dataset=ds_train, eval_dataset=ds_eval, args=args,
)

# ── Auto-resume from latest checkpoint on Drive ──
checkpoints = sorted(
    glob.glob(f"{OUTPUT_DIR}/checkpoint-*"),
    key=lambda x: int(x.split("-")[-1])
)
resume_ckpt = checkpoints[-1] if checkpoints else None

if resume_ckpt:
    step = resume_ckpt.split("-")[-1]
    print(f"RESUMING from {resume_ckpt} (step {step})")
else:
    print(f"Starting fresh — checkpoints save to Drive every {SAVE_STEPS} steps")

total_steps = (len(ds_train) // (BATCH_SIZE * GRAD_ACCUM_STEPS)) * NUM_EPOCHS
print(f"Training: ~{total_steps:,} steps | {NUM_EPOCHS} epochs")
print(f"Effective batch: {BATCH_SIZE} x {GRAD_ACCUM_STEPS} = {BATCH_SIZE * GRAD_ACCUM_STEPS}")
print(f"VRAM estimate: ~43 GB / 80 GB")

t0 = time.time()
stats = trainer.train(resume_from_checkpoint=resume_ckpt)
print(f"\nDONE in {(time.time()-t0)/60:.1f}min | Loss: {stats.training_loss:.4f}")

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


🦥 Unsloth: Packing enabled - training is >2x faster and uses less VRAM!
Starting fresh — checkpoints save to Drive every 50 steps
Training: ~1,004 steps | 2 epochs
Effective batch: 8 x 8 = 64
VRAM estimate: ~43 GB / 80 GB


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 14,254 | Num Epochs = 2 | Total steps = 446
O^O/ \_/ \    Batch size per device = 8 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (8 x 8 x 1) = 64
 "-____-"     Trainable parameters = 536,870,912 of 33,298,994,176 (1.61% trained)


Step,Training Loss,Validation Loss
100,0.851736,0.844410
200,0.807104,0.807914


Unsloth: Not an error, but Qwen3ForCausalLM does not accept `num_items_in_batch`.
Using gradient accumulation will be very slightly less accurate.
Read more on gradient accumulation issues here: https://unsloth.ai/blog/gradient


Step,Training Loss,Validation Loss
100,0.851736,0.844410
200,0.807104,0.807914
300,0.756132,0.792999
400,0.752534,0.785079


eval/loss,█▄▂▁
eval/runtime,█▁▁▅
eval/samples_per_second,▁██▃
eval/steps_per_second,▁██▅
train/epoch,▁▁▁▁▂▂▂▂▂▂▃▃▃▃▃▄▄▄▄▄▅▅▅▅▅▆▆▆▆▆▆▇▇▇▇▇▇███
train/global_step,▁▁▁▁▂▂▂▂▂▂▃▃▃▃▃▄▄▄▄▄▅▅▅▅▅▅▆▆▆▆▆▆▇▇▇▇▇███
train/grad_norm,█▄▁▁▁▂▂▃▂▃▂▁▁▁▁▁▁▂▂▂▁▂▁▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▂▁
train/learning_rate,▄▇███████▇▇▇▇▇▆▆▆▆▅▅▅▄▄▄▄▃▃▃▃▂▂▂▂▂▁▁▁▁▁▁
train/loss,█▅▃▃▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
eval/loss,0.78508
eval/runtime,484.7543



DONE in 972.6min | Loss: 0.8269


---
## 6. Push to HF Hub + GGUF

In [ ]:
# Save LoRA + Push merged
lora_dir = f"{OUTPUT_DIR}/lora_adapter"
model.save_pretrained(lora_dir); tokenizer.save_pretrained(lora_dir)

print(f"Pushing merged model to {HF_REPO_ID}...")
model.push_to_hub_merged(HF_REPO_ID, tokenizer, save_method="merged_16bit", token=True)
print(f"Live: https://huggingface.co/{HF_REPO_ID}")

print(f"\nExporting GGUFs to {HF_GGUF_REPO}...")
for q in GGUF_QUANTS:
    print(f"  {q}...", end=" ")
    model.push_to_hub_gguf(HF_GGUF_REPO, tokenizer, quantization_method=q, token=True)
    print("done")
print(f"GGUFs: https://huggingface.co/{HF_GGUF_REPO}")

Pushing merged model to Shinzmann/naija-petro...


config.json:   0%|          | 0.00/754 [00:00<?, ?B/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...aija-petro/tokenizer.json:   0%|          | 28.8kB / 11.4MB            

...

  ....BF16-00002-of-00002.gguf:   1%|          | 80.0MB / 15.7GB            

No files have been modified since last commit. Skipping to prevent empty commit.


Uploading config.json...


No files have been modified since last commit. Skipping to prevent empty commit.


Uploading Ollama Modelfile...
Unsloth: Successfully uploaded GGUF to https://huggingface.co/Shinzmann/naija-petro-GGUF
Unsloth: Cleaning up temporary files...
done
GGUFs: https://huggingface.co/Shinzmann/naija-petro-GGUF


---
## 7. Create Model Card

In [ ]:
from huggingface_hub import HfApi
import textwrap

model_size = BASE_MODEL.split("-")[-1]

card = textwrap.dedent(f'''\
---
license: apache-2.0
language: [en]
tags: [petroleum-engineering, oil-and-gas, fine-tuned, unsloth, qwen3, naija-petro]
base_model: Qwen/Qwen3-{model_size}
datasets: [custom-petroleum-engineering-20k]
pipeline_tag: text-generation
---

# Naija-Petro -- Petroleum Engineering AI

**Domain-specific LLM** fine-tuned for petroleum engineering on Qwen3-{model_size}.

## Overview
- 20,000 synthetic instruction-response pairs (NVIDIA Data Designer)
- QLoRA fine-tuning with Unsloth (2x faster, 70% less VRAM)
- Covers: drilling, reservoir, production, completions, EOR, well testing

## Quick Start
```python
from transformers import AutoModelForCausalLM, AutoTokenizer
model = AutoModelForCausalLM.from_pretrained("{HF_REPO_ID}", device_map="auto")
tokenizer = AutoTokenizer.from_pretrained("{HF_REPO_ID}")
```

## Ollama
```bash
ollama run hf.co/{HF_GGUF_REPO}:Q4_K_M
```

## Training
| Param | Value |
|---|---|
| Base | Qwen3-{model_size} |
| Method | QLoRA 4-bit |
| LoRA r / alpha | {LORA_R} / {LORA_ALPHA} |
| LR | {LEARNING_RATE} |
| Epochs | {NUM_EPOCHS} |
| Samples | ~19K train / ~1K eval |

## Limitations
- Validate outputs with qualified engineers before operational use
- English only; not for general chat
''')

HfApi().upload_file(path_or_fileobj=card.encode(), path_in_repo="README.md",
    repo_id=HF_REPO_ID, repo_type="model")
print(f"Model card pushed: https://huggingface.co/{HF_REPO_ID}")

No files have been modified since last commit. Skipping to prevent empty commit.


Model card pushed: https://huggingface.co/Shinzmann/naija-petro


---
## 8. Create HF Space (Gradio Chat)

In [ ]:
from huggingface_hub import HfApi

app_code = f'''import gradio as gr
import spaces
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODEL_ID = "{HF_REPO_ID}"
SYSTEM = "{SYSTEM_PROMPT}"

# Load model on first call (cached after that)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
)

print(f"Loading {{MODEL_ID}}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
)
print("Model loaded!")

@spaces.GPU
def respond(message, history):
    messages = [{{"role": "system", "content": SYSTEM}}]
    for msg in history:
        messages.append(msg)
    messages.append({{"role": "user", "content": message}})

    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True, enable_thinking=False
    )
    inputs = tokenizer(text, return_tensors="pt").to(model.device)

    with torch.no_grad():
        output = model.generate(
            **inputs, max_new_tokens=1024,
            temperature=0.4, top_p=0.9, do_sample=True,
            repetition_penalty=1.1,
        )
    response = tokenizer.decode(
        output[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True
    )
    return response

EXAMPLES = [
    "Explain the material balance equation for an undersaturated reservoir.",
    "What are the screening criteria for CO2 EOR?",
    "How do you interpret a Horner plot?",
    "Compare ESP vs gas lift for artificial lift.",
    "What causes water coning and how to mitigate it?",
]

demo = gr.ChatInterface(
    fn=respond,
    title="Naija-Petro -- Petroleum Engineering AI",
    description="Ask about drilling, reservoir, production, completions, EOR, well testing. First response may take ~60s while GPU loads.",
    examples=EXAMPLES,
    theme=gr.themes.Soft(),
    type="messages",
    cache_examples=False,
)

if __name__ == "__main__":
    demo.launch()
'''

requirements = """gradio>=5.0.0
torch
transformers
accelerate
bitsandbytes
spaces
"""

space_readme = f'''---
title: Naija-Petro Chat
emoji: "\\U0001f6e2"
colorFrom: red
colorTo: red
sdk: gradio
sdk_version: "5.36.2"
app_file: app.py
pinned: true
license: apache-2.0
short_description: Petroleum Engineering AI
---
# Naija-Petro Chat
Petroleum engineering AI assistant powered by [{HF_REPO_ID}](https://huggingface.co/{HF_REPO_ID})
'''

api = HfApi()
for name, content in [
    ("app.py", app_code),
    ("requirements.txt", requirements),
    ("README.md", space_readme),
]:
    api.upload_file(
        path_or_fileobj=content.encode(),
        path_in_repo=name,
        repo_id=HF_SPACE_REPO,
        repo_type="space"
    )
    print(f"  Uploaded {{name}}")

print(f"\nSpace rebuilding: https://huggingface.co/spaces/{HF_SPACE_REPO}")
print("First load will take ~2-3 min (downloading + loading 32B model in 4-bit)")

  Uploaded {name}
  Uploaded {name}
  Uploaded {name}

Space rebuilding: https://huggingface.co/spaces/Shinzmann/naija-petro-chat
First load will take ~2-3 min (downloading + loading 32B model in 4-bit)


---
## 9. Evaluation

In [ ]:
import gc
import torch

# ============================================================
# Clear ALL previous models from GPU
# ============================================================
for var_name in ['model', 'trainer', 'base_model', 'bm']:
    if var_name in dir():
        try:
            exec(f'del {var_name}')
        except:
            pass
gc.collect()
torch.cuda.empty_cache()

vram_used = torch.cuda.memory_allocated() / 1e9
print(f"VRAM after cleanup: {vram_used:.1f} GB (should be ~0)")

# ============================================================
# Load fine-tuned model from HuggingFace
# ============================================================
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
)

print(f"\nLoading fine-tuned model: {HF_REPO_ID}...")
tokenizer = AutoTokenizer.from_pretrained(HF_REPO_ID)
model = AutoModelForCausalLM.from_pretrained(
    HF_REPO_ID,
    quantization_config=bnb_config,
    device_map="auto",
)
print(f"✓ Loaded {HF_REPO_ID} | VRAM: {torch.cuda.memory_allocated()/1e9:.1f}GB")

VRAM after cleanup: 18.3 GB (should be ~0)

Loading fine-tuned model: Shinzmann/naija-petro...


Loading weights:   0%|          | 0/707 [00:00<?, ?it/s]

✓ Loaded Shinzmann/naija-petro | VRAM: 38.9GB


In [ ]:
PETRO_EVAL = [
    {"q":"Explain overbalanced vs underbalanced drilling.","cat":"drilling"},
    {"q":"What is kill weight mud and how to calculate it?","cat":"drilling"},
    {"q":"Describe differential sticking mechanisms and prevention.","cat":"drilling"},
    {"q":"What factors determine drill bit selection?","cat":"drilling"},
    {"q":"Explain ECD and its significance.","cat":"drilling"},
    {"q":"Derive the material balance equation for undersaturated reservoir.","cat":"reservoir"},
    {"q":"Explain Buckley-Leverett theory and assumptions.","cat":"reservoir"},
    {"q":"Darcy vs non-Darcy flow in porous media?","cat":"reservoir"},
    {"q":"How to estimate OOIP using volumetric method?","cat":"reservoir"},
    {"q":"Explain relative permeability curves.","cat":"reservoir"},
    {"q":"Describe Vogel IPR vs straight-line IPR.","cat":"production"},
    {"q":"What causes water coning and remedies?","cat":"production"},
    {"q":"Artificial lift selection criteria?","cat":"production"},
    {"q":"How does Nodal Analysis work?","cat":"production"},
    {"q":"GOR behavior in solution gas drive reservoir?","cat":"production"},
    {"q":"Openhole vs cased-hole completions comparison.","cat":"completions"},
    {"q":"Explain hydraulic fracturing and proppant role.","cat":"completions"},
    {"q":"What is skin factor and how to determine it?","cat":"completions"},
    {"q":"Matrix acidizing: sandstone vs carbonate.","cat":"completions"},
    {"q":"Perforation density and phasing selection factors?","cat":"completions"},
    {"q":"Compare thermal, chemical, and miscible gas EOR.","cat":"eor"},
    {"q":"Explain MMP and how it is determined.","cat":"eor"},
    {"q":"What is SAGD and suitable reservoir types?","cat":"eor"},
    {"q":"Polymer flooding mechanism vs waterflooding?","cat":"eor"},
    {"q":"CO2 injection screening criteria?","cat":"eor"},
    {"q":"Horner plot method for buildup analysis.","cat":"well_testing"},
    {"q":"What info from a derivative plot?","cat":"well_testing"},
    {"q":"Flow regimes in a drawdown test.","cat":"well_testing"},
    {"q":"Superposition in multi-rate well testing.","cat":"well_testing"},
    {"q":"What is a DST and what info does it provide?","cat":"well_testing"},
]

try:
    from datasets import load_dataset as hf_load
    ds_bench = hf_load("GainEnergy/oilandgas-engineering-dataset", split="train")
    print(f"GainEnergy benchmark: {len(ds_bench):,} samples")
    HAS_BENCHMARK = True
except:
    HAS_BENCHMARK = False
    print("GainEnergy benchmark not available")

print(f"Custom eval: {len(PETRO_EVAL)} questions, {len(set(q['cat'] for q in PETRO_EVAL))} categories")

GainEnergy benchmark: 453 samples
Custom eval: 30 questions, 6 categories


In [ ]:
import time

def gen(mdl, tok, q, max_tok=512):
    msgs = [{"role":"system","content":SYSTEM_PROMPT},{"role":"user","content":q}]
    txt = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True, enable_thinking=False)
    inp = tok(txt, return_tensors="pt").to(mdl.device)
    with torch.no_grad():
        out = mdl.generate(**inp, max_new_tokens=max_tok, temperature=0.3, top_p=0.9, do_sample=True, repetition_penalty=1.1)
    return tok.decode(out[0][inp["input_ids"].shape[1]:], skip_special_tokens=True).strip()

print("Quick test:", gen(model, tokenizer, "What is porosity?", 100)[:200])

print("\nEvaluating FINE-TUNED model...")
ft_res = []
for i, item in enumerate(PETRO_EVAL):
    t0 = time.time()
    r = gen(model, tokenizer, item["q"])
    ft_res.append({"q":item["q"],"cat":item["cat"],"r":r,"w":len(r.split()),"t":round(time.time()-t0,2)})
    if (i+1)%10==0: print(f"  {i+1}/{len(PETRO_EVAL)}")
print(f"FT avg: {sum(x['w'] for x in ft_res)/len(ft_res):.0f} words")

print(ft_res)

Quick test: **Porosity ( ϕ )** – the fraction of a rock’s bulk volume that consists of void space capable of holding fluids. It is expressed as a dimensionless number or percentage:

\[
\phi = \frac{V_{\text{void

Evaluating FINE-TUNED model...
  10/30
  20/30
  30/30
FT avg: 251 words


In [ ]:
# Unload fine-tuned model
del model
gc.collect()
torch.cuda.empty_cache()
print(f"VRAM after unload: {torch.cuda.memory_allocated()/1e9:.1f} GB")

# Load base model
print("\nLoading base model: Qwen/Qwen3-32B...")
base_tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen3-32B")
base_model = AutoModelForCausalLM.from_pretrained(
    "Qwen/Qwen3-32B",
    quantization_config=bnb_config,
    device_map="auto",
)
print(f"✓ Loaded base | VRAM: {torch.cuda.memory_allocated()/1e9:.1f}GB")

print("\nEvaluating BASE model...")
base_res = []
for i, item in enumerate(PETRO_EVAL):
    t0 = time.time()
    r = gen(base_model, base_tokenizer, item["q"])
    base_res.append({"q":item["q"],"cat":item["cat"],"r":r,"w":len(r.split()),"t":round(time.time()-t0,2)})
    if (i+1)%10==0: print(f"  {i+1}/{len(PETRO_EVAL)}")
print(f"Base avg: {sum(x['w'] for x in base_res)/len(base_res):.0f} words")

VRAM after unload: 0.0 GB

Loading base model: Qwen/Qwen3-32B...


config.json:   0%|          | 0.00/728 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 17 files:   0%|          | 0/17 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/707 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

✓ Loaded base | VRAM: 20.7GB

Evaluating BASE model...


In [ ]:
# Unload base, reload fine-tuned for judging
del base_model, base_tokenizer
gc.collect()
torch.cuda.empty_cache()

print("Reloading fine-tuned model for judging...")
tokenizer = AutoTokenizer.from_pretrained(HF_REPO_ID)
model = AutoModelForCausalLM.from_pretrained(
    HF_REPO_ID,
    quantization_config=bnb_config,
    device_map="auto",
)
print(f"✓ Ready for judging | VRAM: {torch.cuda.memory_allocated()/1e9:.1f}GB")

---
## 10. LLM-as-Judge & Results

In [ ]:
import re

JUDGE_TPL = "You are an expert petroleum engineering professor. Score this answer.\nQ: {q}\nA: {a}\nScore 1-5 each:\nTECHNICAL_ACCURACY: <n>\nCOMPLETENESS: <n>\nTERMINOLOGY: <n>"

def judge(mdl, tok, q, a):
    msgs = [{"role":"user","content":JUDGE_TPL.format(q=q, a=a[:800])}]
    txt = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True, enable_thinking=False)
    inp = tok(txt, return_tensors="pt").to(mdl.device)
    with torch.no_grad():
        out = mdl.generate(**inp, max_new_tokens=80, temperature=0.1, do_sample=False)
    resp = tok.decode(out[0][inp["input_ids"].shape[1]:], skip_special_tokens=True)
    scores = {}
    for m in ["TECHNICAL_ACCURACY","COMPLETENESS","TERMINOLOGY"]:
        match = re.search(rf"{m}:\s*(\d)", resp)
        scores[m.lower()] = int(match.group(1)) if match else 3
    return scores

N = min(20, len(PETRO_EVAL))
print(f"Judging {N} responses (using fine-tuned model as judge)...")
ft_sc, base_sc = [], []
for i in range(N):
    q = PETRO_EVAL[i]["q"]
    ft_sc.append(judge(model, tokenizer, q, ft_res[i]["r"]))
    base_sc.append(judge(model, tokenizer, q, base_res[i]["r"]))
    if (i+1)%5==0: print(f"  {i+1}/{N}")
print("Done")

In [ ]:
import pandas as pd

M = ["technical_accuracy","completeness","terminology"]
ft_avg = {m: sum(s[m] for s in ft_sc)/len(ft_sc) for m in M}
base_avg = {m: sum(s[m] for s in base_sc)/len(base_sc) for m in M}

df = pd.DataFrame({
    "Metric": [m.replace("_"," ").title() for m in M]+["Overall"],
    "Base": [f"{base_avg[m]:.2f}" for m in M]+[f"{sum(base_avg.values())/3:.2f}"],
    "Naija-Petro": [f"{ft_avg[m]:.2f}" for m in M]+[f"{sum(ft_avg.values())/3:.2f}"],
    "Delta": [f"+{ft_avg[m]-base_avg[m]:.2f}" for m in M]+[f"+{(sum(ft_avg.values())-sum(base_avg.values()))/3:.2f}"],
})
print("="*60)
print("NAIJA-PETRO vs BASE MODEL")
print("="*60)
print(df.to_string(index=False))

print("\nBy Category:")
cats = {}
for i in range(N):
    c = PETRO_EVAL[i]["cat"]
    cats.setdefault(c,[]).append(sum(ft_sc[i].values())/3)
for c,s in sorted(cats.items()):
    print(f"  {c:20s}: {sum(s)/len(s):.2f}/5.0")

---
## 11. Save Report

In [ ]:
import json, shutil

report = {"model":HF_REPO_ID,"base":BASE_MODEL,
    "training":{"lora_r":LORA_R,"epochs":NUM_EPOCHS,"lr":LEARNING_RATE,"train":len(ds_train),"eval":len(ds_eval)},
    "scores":{"ft":ft_avg,"base":base_avg,"delta":{m:ft_avg[m]-base_avg[m] for m in M}},
}
Path(OUTPUT_DIR).mkdir(exist_ok=True)
with open(f"{OUTPUT_DIR}/eval_report.json","w") as f: json.dump(report,f,indent=2,default=str)
shutil.copy(f"{OUTPUT_DIR}/eval_report.json", str(DRIVE_BASE/"naija_petro_eval.json"))

if USE_WANDB:
    wandb.log({"eval/ft_overall":sum(ft_avg.values())/3,"eval/base_overall":sum(base_avg.values())/3,
        "eval/improvement":(sum(ft_avg.values())-sum(base_avg.values()))/3})
    wandb.finish()
print("Report saved to Drive and W&B")

---
## 12. Deploy & Demo

In [ ]:
print(f'''
NAIJA-PETRO DEPLOYMENT COMPLETE
================================
Model:  https://huggingface.co/{HF_REPO_ID}
GGUF:   https://huggingface.co/{HF_GGUF_REPO}
Demo:   https://huggingface.co/spaces/{HF_SPACE_REPO}

Python: AutoModelForCausalLM.from_pretrained("{HF_REPO_ID}", device_map="auto")
Ollama: ollama run hf.co/{HF_GGUF_REPO}:Q4_K_M
HF API: InferenceClient("{HF_REPO_ID}")

Score (FT):   {sum(ft_avg.values())/3:.2f}/5.0
Score (Base): {sum(base_avg.values())/3:.2f}/5.0
Improvement:  +{(sum(ft_avg.values())-sum(base_avg.values()))/3:.2f}
''')

In [ ]:
# Interactive demo
for q in ["Horner time ratio in buildup analysis?", "Primary vs secondary vs tertiary recovery?", "Calculate BHP from surface pressure in a gas well?"]:
    print(f"\nQ: {q}\n{'-'*40}")
    print(gen(model, tokenizer, q, 300)[:500])